# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshalKushwaha0027/FlyRankAI/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Setup Connection ---
import duckdb
from google.colab import userdata

# Re-establish the connection and secret
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
# ------------------------

# 1. Build the feature vector
# Define the path to the daily performance table
table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# Build features by aggregating the historical window.
# We handle missing values using COALESCE and prevent division by zero for CTR.
query_features = f"""
    SELECT
        content_hash_id,
        client_hash_id,
        COALESCE(SUM(gsc_impressions), 0) as impressions_90d,
        COALESCE(SUM(gsc_clicks), 0) as clicks_90d,
        CASE
            WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0
        END as ctr_90d
    FROM read_parquet('{table_path}')
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id
"""

df_features = con.sql(query_features).df()

print("Feature vector built successfully. Shape:", df_features.shape)
display(df_features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector built successfully. Shape: (331437, 5)


,content_hash_id,client_hash_id,impressions_90d,clicks_90d,ctr_90d
0,content_05597932fe4da067,client_73cda7b4e4f265ea,57.0,0.0,0.000000
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,0.001073
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,0.000000
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,0.000000
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,0.001066


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Feature Documentation:**

*   **`impressions_90d`**:
    *   **Meaning**: The total number of Google Search Console impressions (`gsc_impressions`) the content received during the aggregated time window.
    *   **Missing Values**: Handled directly in the SQL query using `COALESCE(..., 0)`. If a record has no impressions logged, it defaults to zero.
    *   **Available-when**: Yes, this is a strictly historical metric that exists before predicting future performance.
*   **`clicks_90d`**:
    *   **Meaning**: The total number of Google Search Console clicks (`gsc_clicks`) the content accumulated over the time window.
    *   **Missing Values**: Handled via `COALESCE(..., 0)` to ensure nulls are treated as zero clicks.
    *   **Available-when**: Yes, derived entirely from historical logs before the prediction moment.
*   **`ctr_90d`**:
    *   **Meaning**: The aggregate Click-Through Rate (total clicks divided by total impressions) for the time window.
    *   **Missing Values**: Handled using a SQL `CASE` statement. If total impressions are zero, the CTR is forced to 0. This prevents `NaN` values or division-by-zero errors.
    *   **Available-when**: Yes, computed entirely from the historical impression and click metrics.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Attack 1: Future Window Leakage Check
# We must ensure our features only use data from our historical window (March)
# and didn't accidentally leak data from the target future window.

print("--- Leakage Test 1: Date Window Boundary ---")
date_check_query = f"""
    SELECT
        MIN(report_date) as earliest_date,
        MAX(report_date) as latest_date
    FROM read_parquet('{table_path}')
    WHERE month = '2026-03'
"""
date_df = con.sql(date_check_query).df()
display(date_df)

latest_date = date_df['latest_date'].iloc[0]

# FIX: Slice the string to [0:10] to only compare the 'YYYY-MM-DD' portion
if str(latest_date)[:10] <= '2026-03-31':
    print("✅ PASS: No future data leaked. Max date is within the safe historical window.")
else:
    print("❌ FAIL: Future data detected in the feature set.")

# Attack 2: Label-derived columns and product flags
# Let's inspect our final feature columns to guarantee no target labels (like future drops)
# or specific client product flags snuck in through a "SELECT *" wildcard.

print("\n--- Leakage Test 2: Column Inspection ---")
safe_columns = ['content_hash_id', 'client_hash_id', 'impressions_90d', 'clicks_90d', 'ctr_90d']
current_columns = df_features.columns.tolist()

print(f"Current columns: {current_columns}")

unauthorized_cols = [col for col in current_columns if col not in safe_columns]
if not unauthorized_cols:
    print("✅ PASS: All columns are approved historical features. No label leakage or product flags found.")
else:
    print(f"❌ FAIL: Unauthorized columns detected: {unauthorized_cols}")

--- Leakage Test 1: Date Window Boundary ---


,earliest_date,latest_date
0,2026-03-01,2026-03-31


✅ PASS: No future data leaked. Max date is within the safe historical window.

--- Leakage Test 2: Column Inspection ---
Current columns: ['content_hash_id', 'client_hash_id', 'impressions_90d', 'clicks_90d', 'ctr_90d']
✅ PASS: All columns are approved historical features. No label leakage or product flags found.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

client_hash_id (as a predictive feature): While used as a key to group the data, it is excluded from the actual ML model features because the model must learn generalized patterns, not memorize specific clients.

content_hash_id (as a predictive feature): Excluded because it is a unique identifier with high cardinality. It holds no predictive numerical value and would cause the model to overfit.

content_age_days: Excluded because it belongs to a separate dimension table (dim_content), and joining it created unnecessary complexity for this baseline model when the 90-day performance metrics already provide strong predictive signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.